# Run Attack Strength Curve Probe

本 notebook 用于在 Colab 冷启动环境中产出“攻击强度曲线”的基础数据，并聚合生成论文可消费的曲线表。

当前默认模式为 `from_stage_two_existing_records`: 从已经完成的阶段二真实视频 VAE records 中抽取 H.264、H.265、`temporal_crop`、`frame_dropping` 的已有攻击记录，转换为 `attack_strength_event_scores.jsonl`。该模式可以先打通基础数据、聚合表和阶段四图表链路，但它不是完整的多强度 sweep, 因此 shard manifest 会标记 `claim_support_allowed=false`。

后续若实现真实多强度攻击 runner, 可替换本 notebook 中的基础数据生成步骤, 但聚合步骤保持不变。

## User Config

请先检查本配置区。

关键参数:

- `REPO_URL`: Colab 克隆仓库的 HTTPS 地址。
- `REPO_BRANCH`: 需要验证的分支。
- `RESULT_ROOT`: Google Drive 中 `TSTW/results` 的路径。
- `STAGE_TWO_ROOT`: 阶段二聚合结果目录。留空时自动使用最新 `real_video_vae_latent_probe/shard_aggregated`。
- `MAX_RECORDS_PER_GROUP`: 每个 method/attack/split/role 最多抽取多少条 records。调小可快速 smoke, 设为 `None` 可抽取全部。
- `RUN_PAPER_ARTIFACT_REBUILD`: 是否在聚合后重建阶段四论文图表。


In [ ]:
# 可修改配置区
REPO_URL = "https://github.com/RICHAAARC/TSTW-VW.git"  # Colab 冷启动克隆仓库使用的 HTTPS 地址。
REPO_BRANCH = "explicit-sync"  # 需要验证的分支名称, 必须包含本轮新增脚本。
DRIVE_ROOT = "/content/drive/MyDrive"  # Google Drive 挂载后的根目录。
RESULT_ROOT = f"{DRIVE_ROOT}/TSTW/results"  # TSTW 受治理结果根目录。
LOCAL_ROOT = "/content/TSTW_runtime"  # Colab session-local 工作区。
REPO_DIR = f"{LOCAL_ROOT}/repo"  # 仓库克隆到 Colab 本地后的路径。

STAGE_TWO_ROOT = ""  # 可显式填写阶段二 shard_aggregated 目录; 留空时自动发现最新目录。
METHOD_NAMES = ["frame_prc", "tubelet_only", "tubelet_sync"]  # 攻击强度基础数据覆盖的内部正式方法。
ATTACK_NAMES = ["h264_compression", "h265_compression", "temporal_crop", "frame_dropping"]  # 需要形成曲线基础数据的攻击。
MAX_RECORDS_PER_GROUP = 200  # 每个 method/attack/split/role 最多抽取记录数; 如需全量可改为 None。
TARGET_FPR = 0.01  # 聚合表使用的固定目标 FPR。

RUN_REPOSITORY_SMOKE_TESTS = True  # 是否先运行轻量约束测试。
RUN_BASE_RECORD_EXTRACTION = True  # 是否从阶段二 records 生成 attack_strength_event_scores.jsonl。
RUN_ATTACK_STRENGTH_AGGREGATION = True  # 是否聚合 attack_strength_event_scores.jsonl 为曲线表。
RUN_PAPER_ARTIFACT_REBUILD = True  # 是否重建 paper_artifact_gate 以生成论文图表。
PAPER_SKIP_FIGURES = False  # 是否跳过 PNG/PDF 图表导出。


## Mount Google Drive

挂载 Google Drive。跨 notebook 输入必须来自已经落盘的阶段二聚合包。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Prepare Local Workspace And Repository

重建 Colab 本地工作区并克隆指定分支。

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path


def run_command(command, cwd=None, check=True):
    """运行命令并实时打印 stdout/stderr。"""
    print("$", " ".join(map(str, command)), flush=True)
    completed = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if check and completed.returncode != 0:
        raise RuntimeError("命令失败: " + " ".join(map(str, command)))
    return completed

local_root = Path(LOCAL_ROOT)
if local_root.exists():
    shutil.rmtree(local_root)
local_root.mkdir(parents=True, exist_ok=True)

run_command(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, REPO_DIR])
short_commit = run_command(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR).stdout.strip()
print(json.dumps({"repo_dir": REPO_DIR, "short_commit": short_commit}, ensure_ascii=False, indent=2))


## Install Dependencies

安装聚合与绘图所需依赖。

In [ ]:
run_command([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])


## Verify Repository Contract

运行攻击强度曲线相关轻量测试, 确认当前分支包含基础数据生成器、聚合器和阶段四图表入口。

In [ ]:
if RUN_REPOSITORY_SMOKE_TESTS:
    run_command([
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/constraints/test_attack_strength_curve_probe_builder.py",
        "tests/constraints/test_paper_artifact_gate_builder.py",
    ], cwd=REPO_DIR)
else:
    print("已跳过仓库轻量约束测试。")


## Generate Base Attack Strength Records

从阶段二已有 records 生成攻击强度曲线基础 records。

输出目录:

```text
/content/drive/MyDrive/TSTW/results/attack_strength_curve_probe/shard_runs/<RUN_ID>/
```

注意: 该步骤不新增真实攻击强度, 只把阶段二中已有攻击记录转为曲线基础数据。

In [ ]:
base_records_output = None
if RUN_BASE_RECORD_EXTRACTION:
    extraction_args = [
        "scripts/package_results/run_attack_strength_curve_probe_from_stage_two.py",
        "--result-root", RESULT_ROOT,
        "--short-commit", short_commit,
    ]
    if STAGE_TWO_ROOT:
        extraction_args.extend(["--stage-two-root", STAGE_TWO_ROOT])
    if MAX_RECORDS_PER_GROUP is not None:
        extraction_args.extend(["--max-records-per-group", str(MAX_RECORDS_PER_GROUP)])
    for method_name in METHOD_NAMES:
        extraction_args.extend(["--method-name", method_name])
    for attack_name in ATTACK_NAMES:
        extraction_args.extend(["--attack-name", attack_name])
    completed = run_command([sys.executable, *extraction_args], cwd=REPO_DIR)
    base_records_output = json.loads(completed.stdout)
else:
    print("已跳过基础 records 生成。")


## Aggregate Attack Strength Curve

聚合 `attack_strength_event_scores.jsonl`, 生成 TPR@1%FPR 表、AUC 表和 figure data。

In [ ]:
attack_strength_output = None
if RUN_ATTACK_STRENGTH_AGGREGATION:
    aggregation_args = [
        "scripts/package_results/build_attack_strength_curve_probe.py",
        "--result-root", RESULT_ROOT,
        "--target-fpr", str(TARGET_FPR),
        "--short-commit", short_commit,
    ]
    if base_records_output is not None:
        record_path = Path(base_records_output["output_root"]) / "records" / "attack_strength_event_scores.jsonl"
        aggregation_args.extend(["--record-path", str(record_path)])
    completed = run_command([sys.executable, *aggregation_args], cwd=REPO_DIR)
    attack_strength_output = json.loads(completed.stdout)
else:
    print("已跳过攻击强度曲线聚合。")


## Rebuild Paper Artifacts

可选重建阶段四论文图表。若聚合成功, 阶段四会自动发现最新 `attack_strength_curve_probe/shard_aggregated` 并生成 `paper_attack_strength_curves.pdf/png`。

In [ ]:
paper_artifact_output = None
if RUN_PAPER_ARTIFACT_REBUILD:
    paper_args = [
        "scripts/package_results/build_paper_artifact_gate.py",
        "--result-root", RESULT_ROOT,
        "--short-commit", short_commit,
    ]
    if PAPER_SKIP_FIGURES:
        paper_args.append("--skip-figures")
    completed = run_command([sys.executable, *paper_args], cwd=REPO_DIR)
    paper_artifact_output = json.loads(completed.stdout)
else:
    print("已跳过 paper_artifact_gate 重建。")


## Final Summary

请把本 cell 输出的 `output_root` 路径发回本地检查。重点关注:

- `base_records_output.output_root`
- `attack_strength_output.output_root`
- `paper_artifact_output.output_root`


In [ ]:
final_summary = {
    "short_commit": short_commit,
    "base_records_output": base_records_output,
    "attack_strength_output": attack_strength_output,
    "paper_artifact_output": paper_artifact_output,
}
print(json.dumps(final_summary, ensure_ascii=False, indent=2))

if attack_strength_output is not None:
    summary = attack_strength_output.get("summary", {})
    print(json.dumps({
        "attack_strength_record_count": summary.get("record_count"),
        "attack_strength_tpr_row_count": summary.get("tpr_row_count"),
        "complete_attacks": summary.get("complete_attacks"),
        "complete_methods": summary.get("complete_methods"),
        "aggregation_output_root": attack_strength_output.get("output_root"),
    }, ensure_ascii=False, indent=2))
